# 5 - Landmark Fusion Coverage Analysis

Extracts fusion coverage metrics:
1. **ROS2 bag `/diagnostics`** — `Landmark Fusion Sparsity` status (cumulative keyframe counters)
2. **ROS2 bag `/vio/odom` vs `/odom/corrected`** — iSAM2 correction magnitude

In [ ]:
!pip install rosbags pyulog matplotlib numpy pandas

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import rosbag2_py
import rclpy
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message

from rosbags.rosbag2 import Reader
from rosbags.typesys import get_typestore, Stores

try:
    from pyulog import ULog
except ImportError:
    raise ImportError("Install pyulog: pip install pyulog")

In [ ]:
SITES = {
    "Site A": {
        "fusion_bag": "../bags/real/klk/test_manual_1_fusion_diag",
        "decoded_bag": "../bags/real/klk/test_manual_1_decoded",
        "ulog_path": "../px4_related/flight_logs/klk/ManualPilot.ulg",
    },
    "Site B": {
        "fusion_bag": "../bags/real/zenxin/tri_tree_manual_flight_fusion_diag",
        "decoded_bag": "../bags/real/zenxin/tri_tree_manual_flight_decoded",
        "ulog_path": "../px4_related/flight_logs/zenxin/ManualPilot.ulg",
    },
}

VIO_TOPIC = "/mavros/odometry/out"
CORRECTED_TOPIC = "/odom/corrected"

FUSION_STATUS_NAME = "Landmark Fusion Sparsity"
FUSION_HARDWARE_ID = "iSAM2_LandmarkFusion"

In [ ]:
def _bag_storage_id(bag_path):
    """Detect rosbag2 storage format: 'mcap' or 'sqlite3'."""
    if not os.path.isdir(bag_path):
        return "sqlite3"
    for name in os.listdir(bag_path):
        if name.endswith(".mcap"):
            return "mcap"
    return "sqlite3"


def read_fusion_diagnostics(bag_path):
    """Parse cumulative Landmark Fusion Sparsity diagnostics from a ROS2 bag.
    Returns a list of dicts (one per diagnostic message) with timestamp and all fields."""
    try:
        rclpy.init()
    except Exception:
        pass

    storage_id = _bag_storage_id(bag_path)
    storage_options = rosbag2_py.StorageOptions(uri=bag_path, storage_id=storage_id)
    converter_options = rosbag2_py.ConverterOptions(
        input_serialization_format='cdr',
        output_serialization_format='cdr'
    )
    reader = rosbag2_py.SequentialReader()
    reader.open(storage_options, converter_options)

    topic_types = reader.get_all_topics_and_types()
    type_map = {t.name: t.type for t in topic_types}

    records = []
    while reader.has_next():
        topic, data, timestamp = reader.read_next()
        if topic != '/diagnostics':
            continue

        msg_type = get_message(type_map[topic])
        msg = deserialize_message(data, msg_type)

        for status in msg.status:
            if status.name == FUSION_STATUS_NAME or status.hardware_id == FUSION_HARDWARE_ID:
                entry = {'timestamp_ns': timestamp, 'timestamp_s': timestamp / 1e9}
                for kv in status.values:
                    entry[kv.key] = kv.value
                entry['message'] = status.message
                records.append(entry)

    return records


def read_odom_xy_ts(bag_path, topic):
    """Read (timestamps_s, x, y) from a nav_msgs/Odometry topic in a ROS2 bag."""
    typestore = get_typestore(Stores.ROS2_HUMBLE)
    ts_list, xs, ys = [], [], []
    with Reader(Path(bag_path)) as reader:
        conns = [c for c in reader.connections if c.topic == topic]
        if not conns:
            raise KeyError(f"Topic {topic!r} not found in {bag_path}")
        for conn, ts, rawdata in reader.messages(connections=conns):
            msg = typestore.deserialize_cdr(rawdata, conn.msgtype)
            ts_list.append(ts / 1e9)
            xs.append(msg.pose.pose.position.x)
            ys.append(msg.pose.pose.position.y)
    return np.array(ts_list), np.array(xs), np.array(ys)


def compute_correction_magnitude(decoded_bag, fusion_bag):
    """Compute ||corrected_xy - vio_xy|| over time.
    VIO comes from the decoded bag, corrected odom from the fusion_diag bag."""
    ts_vio, x_vio, y_vio = read_odom_xy_ts(decoded_bag, VIO_TOPIC)
    ts_cor, x_cor, y_cor = read_odom_xy_ts(fusion_bag, CORRECTED_TOPIC)

    x_cor_interp = np.interp(ts_vio, ts_cor, x_cor)
    y_cor_interp = np.interp(ts_vio, ts_cor, y_cor)

    correction = np.sqrt((x_cor_interp - x_vio)**2 + (y_cor_interp - y_vio)**2)
    return ts_vio, correction

In [ ]:
# ── Extract and summarize per site ──

results = {}

for site_name, paths in SITES.items():
    print(f"\n{'='*60}")
    print(f"  {site_name}")
    print(f"{'='*60}")

    fusion_bag = paths['fusion_bag']
    decoded_bag = paths['decoded_bag']

    # 1. Fusion diagnostics (from fusion_diag bag)
    diag = read_fusion_diagnostics(fusion_bag)
    if not diag:
        print(f"  WARNING: No Landmark Fusion Sparsity messages found in {fusion_bag}")
        continue

    final = diag[-1]
    total_kf = int(final.get('Total keyframes', 0))
    lm_kf = int(final.get('Keyframes with landmarks', 0))
    vio_kf = int(final.get('VIO-only keyframes', 0))
    sparse_kf = int(final.get('Sparse-skipped keyframes', 0))
    lm_pct = float(final.get('Landmark keyframe %', 0))
    total_lm_factors = int(final.get('Total landmark factors', 0))
    avg_lm = float(final.get('Avg landmarks/keyframe (when present)', 0))
    known_lm = int(final.get('Known landmarks', 0))

    print(f"  Total keyframes:             {total_kf}")
    print(f"  Keyframes with landmarks:    {lm_kf} ({lm_pct:.1f}%)")
    print(f"  VIO-only keyframes:          {vio_kf}")
    print(f"  Sparse-skipped keyframes:    {sparse_kf}")
    print(f"  Total landmark factors:      {total_lm_factors}")
    print(f"  Avg landmarks/KF (present):  {avg_lm:.1f}")
    print(f"  Known landmarks:             {known_lm}")

    # Reconstruct per-keyframe fusion state from cumulative deltas
    kf_has_lm = []
    kf_timestamps = []
    prev_lm_kf = 0
    for rec in diag:
        cur_lm_kf = int(rec.get('Keyframes with landmarks', 0))
        cur_total = int(rec.get('Total keyframes', 0))
        lm_count = int(rec.get('Last KF landmark count', 0))
        kf_has_lm.append(lm_count > 0)
        kf_timestamps.append(rec['timestamp_s'])

    kf_timestamps = np.array(kf_timestamps)
    kf_has_lm = np.array(kf_has_lm)

    # Count VIO-only segments (contiguous runs of kf_has_lm == False)
    vio_only_mask = ~kf_has_lm
    segment_starts = np.where(np.diff(np.concatenate(([0], vio_only_mask.astype(int)))) == 1)[0]
    segment_ends = np.where(np.diff(np.concatenate((vio_only_mask.astype(int), [0]))) == -1)[0]
    n_vio_segments = len(segment_starts)

    if n_vio_segments > 0 and len(kf_timestamps) > 1:
        seg_durations = []
        for s, e in zip(segment_starts, segment_ends):
            dur = kf_timestamps[min(e, len(kf_timestamps)-1)] - kf_timestamps[s]
            seg_durations.append(dur)
        seg_durations = np.array(seg_durations)
        mean_vio_seg = float(np.mean(seg_durations))
        max_vio_seg = float(np.max(seg_durations))
    else:
        mean_vio_seg, max_vio_seg = 0.0, 0.0

    print(f"  VIO-only segments:           {n_vio_segments}")
    print(f"  Mean VIO-only seg duration:  {mean_vio_seg:.1f} s")
    print(f"  Max VIO-only seg duration:   {max_vio_seg:.1f} s")

    # 2. iSAM2 correction magnitude (VIO from decoded bag, corrected from fusion bag)
    try:
        ts_corr, corr_mag = compute_correction_magnitude(decoded_bag, fusion_bag)
        mean_corr = float(np.mean(corr_mag))
        std_corr = float(np.std(corr_mag))
        print(f"  Mean iSAM2 correction:       {mean_corr:.3f} ± {std_corr:.3f} m")
    except Exception as e:
        print(f"  iSAM2 correction: FAILED ({e})")
        ts_corr, corr_mag = None, None
        mean_corr, std_corr = 0.0, 0.0

    results[site_name] = {
        'total_kf': total_kf, 'lm_kf': lm_kf, 'vio_kf': vio_kf,
        'sparse_kf': sparse_kf, 'lm_pct': lm_pct,
        'avg_lm': avg_lm, 'known_lm': known_lm,
        'n_vio_segments': n_vio_segments,
        'mean_vio_seg': mean_vio_seg, 'max_vio_seg': max_vio_seg,
        'mean_corr': mean_corr, 'std_corr': std_corr,
        'kf_timestamps': kf_timestamps, 'kf_has_lm': kf_has_lm,
        'ts_corr': ts_corr, 'corr_mag': corr_mag,
    }

print("\nDone.")

In [ ]:
# ── Summary table ──

if not results:
    print("No results to display. Make sure bags contain Landmark Fusion Sparsity diagnostics.")
    print("Re-record bags using: ros2 launch uosm_uav_bringup test_landmark_fusion.launch.py record:=true")
else:
    rows = []
    for site_name, r in results.items():
        rows.append({
            'Site': site_name,
            'Total Keyframes': r['total_kf'],
            'Landmark-Fused KF': r['lm_kf'],
            'VIO-Only KF': r['vio_kf'],
            'Landmark KF %': f"{r['lm_pct']:.1f}",
            'Avg LM / KF (present)': f"{r['avg_lm']:.1f}",
            'Known Landmarks': r['known_lm'],
            'VIO-Only Segments': r['n_vio_segments'],
            'Mean VIO-Only Seg (s)': f"{r['mean_vio_seg']:.1f}",
            'Max VIO-Only Seg (s)': f"{r['max_vio_seg']:.1f}",
            'Mean Correction (m)': f"{r['mean_corr']:.3f} ± {r['std_corr']:.3f}",
        })

    df = pd.DataFrame(rows).set_index('Site').T
    print(df.to_string())

In [ ]:
# ── Trajectory colored by localization mode ──

if not results:
    print("No results to plot.")
else:
    fig, axes = plt.subplots(1, len(results), figsize=(7 * len(results), 6))
    if len(results) == 1:
        axes = [axes]

    for ax, (site_name, r) in zip(axes, results.items()):
        fusion_bag = SITES[site_name]['fusion_bag']
        decoded_bag = SITES[site_name]['decoded_bag']
        try:
            ts_odom, x_odom, y_odom = read_odom_xy_ts(fusion_bag, CORRECTED_TOPIC)
        except Exception:
            ts_odom, x_odom, y_odom = read_odom_xy_ts(decoded_bag, VIO_TOPIC)

        kf_ts = r['kf_timestamps']
        kf_lm = r['kf_has_lm'].astype(float)
        if len(kf_ts) > 1:
            fusion_state = np.interp(ts_odom, kf_ts, kf_lm)
        else:
            fusion_state = np.zeros_like(ts_odom)

        lm_mask = fusion_state > 0.5
        vio_mask = ~lm_mask

        ax.scatter(x_odom[vio_mask], y_odom[vio_mask], c='#ff7f0e', s=2, alpha=0.6, label='VIO-only')
        ax.scatter(x_odom[lm_mask], y_odom[lm_mask], c='#2ca02c', s=2, alpha=0.6, label='Landmark-fused')
        ax.set_xlabel('X (m)')
        ax.set_ylabel('Y (m)')
        ax.set_title(site_name)
        ax.set_aspect('equal')
        ax.legend(markerscale=5)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# ── iSAM2 correction magnitude over time ──

if not results:
    print("No results to plot.")
else:
    fig, axes = plt.subplots(1, len(results), figsize=(7 * len(results), 4))
    if len(results) == 1:
        axes = [axes]

    for ax, (site_name, r) in zip(axes, results.items()):
        if r['ts_corr'] is not None:
            t = r['ts_corr'] - r['ts_corr'][0]
            ax.plot(t, r['corr_mag'], linewidth=1, alpha=0.8)
            ax.axhline(r['mean_corr'], color='r', linestyle='--', label=f"Mean = {r['mean_corr']:.3f} m")
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Correction magnitude (m)')
            ax.set_title(f'{site_name} — iSAM2 Correction')
            ax.legend()
            ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()